# 02 · NLP Model Evaluation: intent, slots and emergency detection
Civic Operations · Group 23 · runs locally (GPU if available)

> ## ⚠️ READ ME FIRST: every team member goes through this notebook before building their component
> It picks the model behind the **C01 intake bot** and the **C03 emergency safeguard** by benchmarking light models against each other on a **hand-written gold test set**. **Mohammad runs the full version on his local GPU** (the data never leaves our machines); on a laptop set `QUICK = True` for a fast smoke run. Khalid: Section D is the harness for your C02 profanity models.
>
> **Before you start:** `py -m pip install -r requirements-ml.txt` from the repo root.
> **Rule:** do NOT change the shared tables or the synthetic population on your own. If you need a change, say so in
> the team chat first, because these tables feed everyone's component (see `05_Team_Workspace/`).

## Why this design
* **Constraint:** the bot must be *intelligent but not resource heavy* (it serves on a free CPU host) → we compare
  models from a 50 KB TF-IDF model up to a 66M-parameter DistilBERT, and report **accuracy and latency and size** together.
* **Honest testing:** training data are template-generated (`synthetic_nlp_utterances`); the test that counts is
  `data/reference/nlp_gold_test.csv`, 152 hand-written, messy, realistic lines never used in training.
* **Safety:** for emergencies we report **recall** first (a missed 911 case is the worst error), at a fixed precision.

| Task | A0 baseline | A1 | A2 | A3 |
|---|---|---|---|---|
| Intent (37 classes) | keyword rules (current C01) | TF-IDF + LinearSVC | MiniLM sentence embeddings + LogReg | fine-tuned DistilBERT |
| Slots (BIO) | regex (current C01) | – | – | fine-tuned DistilBERT token classifier |
| Emergency | lexicon rules (current C03) | TF-IDF + LogReg | rules OR model | – |

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "01_Application" / "data" / "pipeline").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "01_Application"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 200)
P = ROOT / "01_Application" / "data" / "processed"; R = ROOT / "01_Application" / "data" / "reference"
print("repo:", ROOT)
import time, json, torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support
QUICK = True            # False on the GPU machine for the full benchmark
SEED = 2026
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| quick:", QUICK)

In [ ]:
utt = pd.read_parquet(P / "synthetic_nlp_utterances.parquet")
gold = pd.read_csv(R / "nlp_gold_test.csv")
train, val = utt[utt.split == "train"], utt[utt.split == "val"]
if QUICK:
    train = train.groupby("intent", group_keys=False).apply(lambda g: g.sample(min(len(g), 80), random_state=SEED))
print(len(train), "train |", len(val), "val |", len(gold), "gold (hand-written)")
LABELS = sorted(utt.intent.unique())

In [ ]:
RESULTS = []
def record(task, model, y_true, y_pred, ms_per_item, size_mb, split):
    RESULTS.append(dict(task=task, model=model, split=split, accuracy=accuracy_score(y_true, y_pred),
                        macro_f1=f1_score(y_true, y_pred, average="macro", zero_division=0),
                        ms_per_item=round(ms_per_item, 3), size_mb=round(size_mb, 2)))
    return RESULTS[-1]

def timed(fn, xs):
    t = time.perf_counter(); out = fn(xs); return out, (time.perf_counter() - t) * 1000 / max(len(xs), 1)

## A · Intent classification

### A0 · Keyword baseline (the current C01 placeholder logic)

In [ ]:
import re, csv
# Self-contained keyword baseline: one word-overlap score per request type, with the
# emergency lexicon checked first, exactly as the rule-based intake does.
KEYWORDS = {r["request_type_code"]: r["keywords"] for r in csv.DictReader(open(R / "intent_keywords.csv", encoding="utf-8"))}
EM_PATTERNS = [re.compile(r["pattern"], re.I) for r in csv.DictReader(open(R / "emergency_lexicon.csv", encoding="utf-8"))]
def a0_one(text):
    if any(p.search(text) for p in EM_PATTERNS):
        return "EMERGENCY"
    words = set(re.findall(r"[a-z\-]+", text.lower()))
    scores = {code: len(words & set(kw.split())) for code, kw in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else "OTHER"
def a0(texts):
    return [a0_one(t) for t in texts]
for name, df in [("val", val), ("gold", gold)]:
    pred, ms = timed(a0, df.text.tolist()); print(record("intent", "A0 keywords", df.intent, pred, ms, 0.01, name))

### A1 · TF-IDF (word + character n-grams) + LinearSVC

In [ ]:
from sklearn.pipeline import make_pipeline, make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
import pickle
a1 = make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2),
                              TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, min_df=2)),
                   LinearSVC(C=0.8))
a1.fit(train.text, train.intent)
size = len(pickle.dumps(a1)) / 1e6
for name, df in [("val", val), ("gold", gold)]:
    pred, ms = timed(a1.predict, df.text.tolist()); print(record("intent", "A1 tfidf+svm", df.intent, pred, ms, size, name))

### A2 · MiniLM sentence embeddings (frozen) + Logistic Regression

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)
Xtr = enc.encode(train.text.tolist(), batch_size=128, show_progress_bar=False, normalize_embeddings=True)
a2 = LogisticRegression(max_iter=2000, C=4.0).fit(Xtr, train.intent)
def a2_predict(texts): return a2.predict(enc.encode(texts, batch_size=128, normalize_embeddings=True))
for name, df in [("val", val), ("gold", gold)]:
    pred, ms = timed(a2_predict, df.text.tolist()); print(record("intent", "A2 minilm+logreg", df.intent, pred, ms, 90.9, name))

### A3 · Fine-tuned DistilBERT (GPU recommended)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
MODEL = "distilbert-base-uncased"; EPOCHS = 1 if QUICK else 3
tok = AutoTokenizer.from_pretrained(MODEL)
l2i = {l: i for i, l in enumerate(LABELS)}
a3 = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(LABELS)).to(DEVICE)
opt = torch.optim.AdamW(a3.parameters(), lr=5e-5)
batches = DataLoader(list(zip(train.text, train.intent.map(l2i))), batch_size=32, shuffle=True)
a3.train()
for ep in range(EPOCHS):
    for texts, ys in batches:
        b = tok(list(texts), padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        loss = a3(**b, labels=ys.to(DEVICE)).loss; loss.backward(); opt.step(); opt.zero_grad()
    print("epoch", ep + 1, "loss", round(loss.item(), 4))
a3.eval()
@torch.no_grad()
def a3_predict(texts):
    out = []
    for i in range(0, len(texts), 64):
        b = tok(texts[i:i + 64], padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
        out += [LABELS[j] for j in a3(**b).logits.argmax(-1).tolist()]
    return out
for name, df in [("val", val), ("gold", gold)]:
    pred, ms = timed(a3_predict, df.text.tolist()); print(record("intent", "A3 distilbert-ft", df.intent, pred, ms, 255.0, name))

### Error analysis on the gold set
Read the mistakes: they tell us which intents confuse (e.g. RT021 drain vs RT022 flooding) and which phrasing the templates missed.

In [ ]:
best_pred = a1.predict(gold.text)
err = gold.assign(pred=best_pred)[lambda d: d.pred != d.intent]
print(len(err), "errors of", len(gold)); err[["text", "intent", "pred"]].head(25)

## B · Slot filling (address, cross street, borough, time, hazard)
Span-level F1 on the validation set (the gold set has no span labels). The regex baseline is the current C01 logic.

In [ ]:
def spans(tokens, tags):
    out, cur = set(), None
    for i, t in enumerate(tags):
        if t.startswith("B-"): cur = [t[2:], i, i]; out.add(tuple(cur))
        elif t.startswith("I-") and cur and t[2:] == cur[0]: out.discard(tuple(cur)); cur[2] = i; out.add(tuple(cur))
        else: cur = None
    return out
def span_f1(gold_sets, pred_sets):
    tp = sum(len(g & p) for g, p in zip(gold_sets, pred_sets)); fp = sum(len(p - g) for g, p in zip(gold_sets, pred_sets))
    fn = sum(len(g - p) for g, p in zip(gold_sets, pred_sets)); pr = tp / max(tp + fp, 1); rc = tp / max(tp + fn, 1)
    return dict(precision=round(pr, 3), recall=round(rc, 3), f1=round(2 * pr * rc / max(pr + rc, 1e-9), 3))
sv = val[val.has_slot_labels].head(800 if QUICK else 4000)
gold_sets = [spans(list(t), list(g)) for t, g in zip(sv.tokens, sv.bio_tags)]

In [ ]:
# B0 regex baseline: we only score BOROUGH and ADDRESS token overlap, which is what the regex extracts
import re
def regex_tags(tokens):
    r = c01.run(IntakeParseInput(session_id="nb", utterance=" ".join(tokens))).slots
    tags = ["O"] * len(tokens)
    for lab, val_ in (("ADDRESS", r.incident_address), ("BOROUGH", r.borough.value.lower() if r.borough else None)):
        if not val_: continue
        words = val_.lower().split()
        for i in range(len(tokens) - len(words) + 1):
            if [w.lower().strip(",.") for w in tokens[i:i + len(words)]] == words:
                tags[i] = "B-" + lab; tags[i + 1:i + len(words)] = ["I-" + lab] * (len(words) - 1); break
    return tags
b0 = span_f1(gold_sets, [spans(list(t), regex_tags(list(t))) for t in sv.tokens]); b0

In [ ]:
from transformers import AutoModelForTokenClassification
TAGS = sorted({t for ts in utt[utt.has_slot_labels].bio_tags for t in ts}); t2i = {t: i for i, t in enumerate(TAGS)}
tr = train[train.has_slot_labels]
b1 = AutoModelForTokenClassification.from_pretrained(MODEL, num_labels=len(TAGS)).to(DEVICE)
opt = torch.optim.AdamW(b1.parameters(), lr=5e-5)
def encode(tokens_list, tags_list=None):
    enc_ = tok([list(t) for t in tokens_list], is_split_into_words=True, truncation=True, max_length=64, padding=True, return_tensors="pt")
    if tags_list is None: return enc_
    labels = []
    for i, tags in enumerate(tags_list):
        wid, prev, lab = enc_.word_ids(i), None, []
        for w in wid:
            lab.append(-100 if w is None or w == prev else t2i[tags[w]]); prev = w
        labels.append(lab)
    enc_["labels"] = torch.tensor(labels); return enc_
b1.train()
for ep in range(EPOCHS):
    for i in range(0, len(tr), 32):
        chunk = tr.iloc[i:i + 32]; b = encode(chunk.tokens, chunk.bio_tags).to(DEVICE)
        loss = b1(**b).loss; loss.backward(); opt.step(); opt.zero_grad()
b1.eval()
@torch.no_grad()
def b1_tags(tokens):
    e = encode([tokens]).to(DEVICE); pred = b1(**e).logits.argmax(-1)[0].tolist(); out, prev = ["O"] * len(tokens), None
    for k, w in enumerate(e.word_ids(0)):
        if w is not None and w != prev: out[w] = TAGS[pred[k]]
        prev = w
    return out
b1_score = span_f1(gold_sets, [spans(list(t), b1_tags(list(t))) for t in sv.tokens]); b1_score

## C · Emergency detection (C03): recall first

In [ ]:
# The rules run from the shared lexicon, in this notebook, with no package to install.
NEGATION = re.compile(r"(no one|nobody|not) (is )?(hurt|injured|inside|stuck|trapped)", re.I)
LEX = [(re.compile(r["pattern"], re.I), float(r["weight"])) for r in
       csv.DictReader(open(R / "emergency_lexicon.csv", encoding="utf-8"))]
def rule_hit(text):
    hits = [w for pat, w in LEX if pat.search(text)]
    if not hits:
        return False
    return max(hits) * (0.6 if NEGATION.search(text) else 1.0) > 0
g_is_em = (gold.intent == "EMERGENCY").to_numpy()
rules = np.array([rule_hit(t) for t in gold.text])
from sklearn.linear_model import LogisticRegression as LR
em = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LR(max_iter=2000, class_weight="balanced"))
em.fit(train.text, train.intent == "EMERGENCY")
proba = em.predict_proba(gold.text)[:, 1]
rows = []
for name, pred in [("C0 rules", rules), ("C1 tfidf+logreg @0.5", proba >= 0.5), ("C2 rules OR model@0.35", rules | (proba >= 0.35))]:
    p, r, f, _ = precision_recall_fscore_support(g_is_em, pred, average="binary", zero_division=0)
    fp_hydrant = int(pred[gold.text.str.contains("hydrant|fire escape", case=False).to_numpy()].sum())
    rows.append(dict(model=name, recall=round(r, 3), precision=round(p, 3), f1=round(f, 3), false_alarms_on_hydrant_lines=fp_hydrant))
pd.DataFrame(rows)

In [ ]:
missed = gold[g_is_em & ~(rules | (proba >= 0.35))]
print("Emergencies still missed (must be zero before demo):"); missed[["text", "emergency_category"]]

## D · Profanity (C02): harness for Khalid
Plug your models in here; the gold set has a `profanity` column (NONE / MILD / SEVERE). Report precision, recall and the false-positive rate on clean lines. This notebook deliberately does not implement C02.

In [ ]:
def evaluate_profanity(predict_fn, name):
    y = gold.profanity.ne("NONE").to_numpy(); pred = np.array([bool(predict_fn(t)) for t in gold.text])
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    return dict(model=name, precision=round(p, 3), recall=round(r, 3), f1=round(f, 3), fpr_clean=round(pred[~y].mean(), 3))
# Example once C02 exists:
# evaluate_profanity(my_predict_fn, "C02 v1")

## E · Results and decision

In [ ]:
res = pd.DataFrame(RESULTS)
table = res.pivot_table(index="model", columns="split", values=["accuracy", "macro_f1"]).round(3)
table.columns = [f"{m}_{s}" for m, s in table.columns]
table.join(res.groupby("model")[["ms_per_item", "size_mb"]].first()).sort_values("macro_f1_gold", ascending=False)

**Decision rule (agreed before looking at the numbers):** choose the smallest model whose **gold macro-F1** is within
2 points of the best, provided CPU latency is < 20 ms per utterance. For emergencies choose the variant with **recall = 1.0 on gold**
and the highest precision. Record the choice and the numbers in `05_Team_Workspace/` and the report.

In [ ]:
# Export the chosen artifacts (joblib or HF format) with a manifest, so the API can load exactly this version
import joblib, hashlib, datetime
OUT = ROOT / "models"; OUT.mkdir(exist_ok=True)
joblib.dump(a1, OUT / "c01_intent_tfidf_svm.joblib"); joblib.dump(em, OUT / "c03_emergency_tfidf_lr.joblib")
man = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()[:16] for p in OUT.glob("*.joblib")}
man["_trained_at"] = datetime.datetime.now().isoformat(timespec="seconds"); man["_quick"] = QUICK
import sklearn; man["_sklearn"] = sklearn.__version__
(OUT / "manifest.json").write_text(json.dumps(man, indent=2)); man